In [1]:
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt

In [4]:
customer = pd.read_csv("D:\\Project_Olis\\olist_customers_dataset.csv")
geolocation = pd.read_csv("D:\\Project_Olis\\olist_geolocation_dataset.csv")
order_items= pd.read_csv("D:\\Project_Olis\\olist_order_items_dataset.csv")
payments= pd.read_csv("D:\Project_Olis\olist_order_payments_dataset.csv")
review=pd.read_csv("D:\Project_Olis\olist_order_reviews_dataset.csv")
order=pd.read_csv("D:\Project_Olis\olist_orders_dataset.csv")
products=pd.read_csv("D:\Project_Olis\olist_products_dataset.csv")
sellers=pd.read_csv("D:\Project_Olis\olist_sellers_dataset.csv")
category=pd.read_csv("D:\Project_Olis\product_category_name_translation.csv")

In [5]:
tables = {
    "customer": customer,
    "geolocation":geolocation,
    "order_items": order_items,
    "payments": payments,
    "review": review,
    "order":order,
    "products":products,
    "sellers": sellers,
    "category":category

}

In [6]:
for name, df in tables.items():
    print(f"\n===== {name} =====")

    print("NULL:")
    print(df.isnull().sum())

    print("Duplicate:")
    print(df.duplicated().sum())


===== customer =====
NULL:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplicate:
0

===== geolocation =====
NULL:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Duplicate:
261831

===== order_items =====
NULL:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Duplicate:
0

===== payments =====
NULL:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Duplicate:
0

===== review =====
NULL:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       8

### XỬ LÝ VỚI BẢNG REVIEW

In [15]:

# 1. XÓA DÒNG THIẾU ID
    # review_id và order_id là thông tin định danh quan trọng thiếu một trong hai -> loại dòng
review[['review_id', 'order_id']] = review[['review_id', 'order_id']].replace(
    r'^\s*$',
    pd.NA,
    regex=True
)
review = review.dropna(
    subset=['review_id', 'order_id']
)

#2 fill 
review['review_comment_title'] = review['review_comment_title'].fillna('No Title')
review['review_comment_message'] = review['review_comment_message'].fillna('No Comment')

# 3. Kiểm tra duplicate
print("Duplicate toàn bộ dòng:", review.duplicated().sum())
print("Duplicate review_id:", review['review_id'].duplicated().sum())


#convert dtype
review['review_creation_date'] = pd.to_datetime(
    review['review_creation_date']
).dt.date

review['review_answer_timestamp'] = pd.to_datetime(
    review['review_answer_timestamp']
)

# 5. Kiểm tra kết quả
print("\nShape sau cleaning:", review.shape)

print("\nMissing values:")
print(review.isnull().sum())

print(review.dtypes)
review.head()



Duplicate toàn bộ dòng: 0
Duplicate review_id: 814

Shape sau cleaning: (99224, 7)

Missing values:
review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date               object
review_answer_timestamp    datetime64[us]
dtype: object


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Title,No Comment,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Title,No Comment,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Title,No Comment,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,No Title,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,No Title,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


### XỬ LÝ VỚI BẢNG ORDER

In [16]:

date_cols = [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

# Drop các dòng:
# order_status = delivered
# VÀ có ít nhất 1 cột ngày bị NaN
order = order.drop(
    order[
        (order['order_status'] == 'delivered') &
        (order[date_cols].isna().any(axis=1))
    ].index
)

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

order[date_cols] = order[date_cols].apply(
    pd.to_datetime,
    errors='coerce'
)
print(order.dtypes)
review.head()

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Title,No Comment,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Title,No Comment,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Title,No Comment,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,No Title,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,No Title,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [10]:
category.columns = (
    category.columns
    .astype(str)
    .str.strip()
    .str.replace('"', '', regex=False)
)

print(category.columns.tolist())

['product_category_name', 'product_category_name_english']


### XỬ LÝ VỚI BẢNG PRODUCT

In [12]:
# ==============================
# 5. CHUẨN HÓA CATEGORY
# ==============================

# Chuẩn hóa TÊN CỘT của products (cột đang bị dính khoảng trắng và dấu ngoặc kép,
# ví dụ: ' "product_category_name"' -> 'product_category_name')
products.columns = (
    products.columns
    .astype(str)
    .str.strip()
    .str.replace('"', '', regex=False)
)

# Chuẩn hóa tên cột của category
category.columns = (
    category.columns
    .str.strip()
    .str.replace('"', '', regex=False)
)

# Chuẩn hóa giá trị category ở products
products['product_category_name'] = (
    products['product_category_name']
    .astype(str)
    .str.strip()
    .str.replace('"', '', regex=False)
)

# Chuẩn hóa giá trị category ở bảng dịch
category['product_category_name'] = (
    category['product_category_name']
    .astype(str)
    .str.strip()
    .str.replace('"', '', regex=False)
)

# Chuẩn hóa tên cột English
category['product_category_name_english'] = (
    category['product_category_name_english']
    .astype(str)
    .str.strip()
    .str.replace('"', '', regex=False)
)


# ==============================
# 6. BỔ SUNG CATEGORY THIẾU
# ==============================

manual_map = pd.DataFrame({
    'product_category_name': [
        'pc_gamer',
        'portateis_cozinha_e_preparadores_de_alimentos',
        'unknown'
    ],
    'product_category_name_english': [
        'pc_gamer',
        'portable_kitchen_food_preparators',
        'unknown'
    ]
})


cat_trans_full = pd.concat(
    [
        category[
            [
                'product_category_name',
                'product_category_name_english'
            ]
        ],
        manual_map
    ],
    ignore_index=True
)


# ==============================
# 7. XÓA CỘT ENGLISH CŨ
# ==============================

products = products.drop(
    columns=[
        col for col in products.columns
        if col == 'product_category_name_english'
    ],
    errors='ignore'
)


# ==============================
# 8. MERGE
# ==============================

products = products.merge(
    cat_trans_full,
    on='product_category_name',
    how='left'
)


# Category không tìm thấy
products['product_category_name_english'] = (
    products['product_category_name_english']
    .fillna('unknown')
)


# ==============================
# 9. KIỂM TRA
# ==============================

print("Shape:", products.shape)

print("\nSố category:")
print(products['product_category_name'].nunique())

print("\nSố category English:")
print(products['product_category_name_english'].nunique())

print("\nMissing:")
print(products['product_category_name_english'].isna().sum())

print("\nMột số category:")
print(
    products[
        [
            'product_category_name',
            'product_category_name_english'
        ]
    ].drop_duplicates().head(20)
)

Shape: (32951, 10)

Số category:
74

Số category English:
74

Missing:
0

Một số category:
               product_category_name product_category_name_english
0                         perfumaria                     perfumery
1                              artes                           art
2                      esporte_lazer                sports_leisure
3                              bebes                          baby
4              utilidades_domesticas                    housewares
5              instrumentos_musicais           musical_instruments
6                         cool_stuff                    cool_stuff
7                   moveis_decoracao               furniture_decor
8                   eletrodomesticos               home_appliances
9                         brinquedos                          toys
10                   cama_mesa_banho                bed_bath_table
14  construcao_ferramentas_seguranca     construction_tools_safety
17            informatica_acessorios  

In [13]:
print(products['product_category_name'].head(10))
print(category['product_category_name'].head(10))

0               perfumaria
1                    artes
2            esporte_lazer
3                    bebes
4    utilidades_domesticas
5    instrumentos_musicais
6               cool_stuff
7         moveis_decoracao
8         eletrodomesticos
9               brinquedos
Name: product_category_name, dtype: str
0              beleza_saude
1    informatica_acessorios
2                automotivo
3           cama_mesa_banho
4          moveis_decoracao
5             esporte_lazer
6                perfumaria
7     utilidades_domesticas
8                 telefonia
9        relogios_presentes
Name: product_category_name, dtype: str


### XỬ LÝ VỚI BẢNG ORDER_ITEMS

In [17]:
order_items['order_item_id'] = order_items['order_item_id'].astype(str)
order_items['shipping_limit_date']=order_items['shipping_limit_date'].astype('datetime64[us]')

### XỬ LÝ VỚI BẢNG GEOLOCATION

In [18]:
# Xu ly duplicate của geolocation
geolocation = geolocation.drop_duplicates().reset_index(drop=True)

In [19]:
tables = {
    "customer": customer,
    "geolocation":geolocation,
    "order_items": order_items,
    "payments": payments,
    "review": review,
    "order":order,
    "products":products,
    "sellers": sellers,
    "category":category

}

In [20]:
for name, df in tables.items():
    print(f"\n===== {name} =====")

    print("NULL:")
    print(df.isnull().sum())

    print("Duplicate:")
    print(df.duplicated().sum())

    print(df.dtypes)


===== customer =====
NULL:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplicate:
0
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

===== geolocation =====
NULL:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Duplicate:
0
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

===== order_items =====
NULL:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0


In [14]:
import os

output_dir = "D:\\Project_Olis\\cleaned"

# Tạo folder nếu chưa tồn tại
os.makedirs(output_dir, exist_ok=True)

# Lưu riêng products
output_path = os.path.join(
    output_dir,
    "products_cleaned.csv"
)

products.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã lưu: {output_path}")
print(f"Số dòng: {products.shape[0]}")
print(f"Số cột: {products.shape[1]}")

Đã lưu: D:\Project_Olis\cleaned\products_cleaned.csv
Số dòng: 32951
Số cột: 10
